In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import torch
import torch.nn as nn
import numpy as np
import scipy
import math
import itertools
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from torch.distributions.normal import Normal
from scipy.interpolate import griddata
%matplotlib inline

In [ ]:

# 标准正态分布
normal = Normal(0, 1)

# 定义神经网络
class simple_NN2(nn.Module):
    def __init__(self):
        super(simple_NN2, self).__init__()
        self.linear_tanh_stack = nn.Sequential(
            nn.Linear(2, 16),  # 2 inputs: A, t
            nn.Tanh(),
            nn.Linear(16, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1),
        )

    def forward(self, A, t):
        x_stack = torch.cat([A, t], dim=1)  # concatenate A and t
        out = self.linear_tanh_stack(x_stack)
        return out


# 自动求导工具
def df(output: torch.Tensor, input_var: torch.Tensor, order: int = 1) -> torch.Tensor:
    df_value = output
    for _ in range(order):
        df_value = torch.autograd.grad(
            df_value,
            input_var,
            grad_outputs=torch.ones_like(df_value),
            create_graph=True,
            retain_graph=True,
        )[0]
    return df_value


def dfdt(model: simple_NN2, A: torch.Tensor, t: torch.Tensor, order: int = 1):
    f_value = model(A, t)
    return df(f_value, t, order=order)


def dfdx(model: simple_NN2, A: torch.Tensor, t: torch.Tensor, order: int = 1):
    f_value = model(A, t)
    return df(f_value, A, order=order)


# 定义终端条件
def final_condition(A, K):
    """
    Compute the final condition: C_τ = max(A_τ - K, 0)
    """
    return torch.maximum(A - K, torch.tensor(0.0, device=A.device)).reshape(-1, 1)


# PDE损失函数
def pde_loss(model, A, t, pi, beta, alpha, r, T, tau):
    """
    Compute the PDE loss for the carbon option price.
    """
    # 时间导数 ∂C/∂t
    dC_dt = dfdt(model, A, t, order=1)

    # 空间二阶导数 ∂²C/∂A²
    d2C_dA2 = dfdx(model, A, t, order=2)

    # 计算 z_t
    z_t = beta * (T - t).pow(-alpha)

    # 计算 Φ⁻¹(A_t / π)
    A_pi_ratio = A / pi
    Phi_inv = normal.icdf(A_pi_ratio.clamp(min=1e-6, max=1-1e-6))  # 限制输入值避免溢出

    # 计算 Φ'(Φ⁻¹(A_t / π))
    Phi_prime = torch.exp(-0.5 * Phi_inv.pow(2)) / (torch.sqrt(torch.tensor(2.0 * torch.pi)))

    # 非线性项
    nonlinear_term = 0.5 * d2C_dA2 * (pi * Phi_prime * torch.sqrt(z_t)).pow(2)

    # PDE残差
    pde_residual = dC_dt + nonlinear_term - r * model(A, t)

    # 返回损失
    return pde_residual.pow(2).mean()


# 总损失函数
def compute_loss(
    model: simple_NN2,
    A: torch.Tensor,
    t: torch.Tensor,
    A_idx: torch.Tensor,
    t_idx: torch.Tensor,
    pi: float,
    beta: float,
    alpha: float,
    r: float,
    T: float,
    tau: float,
    K: float,
    device: str,
) -> torch.float:
    # PDE损失
    pde_loss_val = pde_loss(model, A, t, pi, beta, alpha, r, T, tau)

    # 边界条件
    boundary_A0 = torch.ones_like(t_idx, requires_grad=True).to(device) * A[0]
    boundary_loss_A0 = model(boundary_A0, t_idx)  # f(A0, t)

    boundary_A1 = torch.ones_like(t_idx, requires_grad=True).to(device) * A[-1]
    boundary_loss_A1 = model(boundary_A1, t_idx) - (A[-1] - K * torch.exp(-r * (T - t_idx)))

    # boundary_loss_A1 = model(boundary_A1, t_idx)  # 假设值接近0

    # 终端条件
    f_final = final_condition(A_idx, K)
    t_final = torch.ones_like(A_idx, device=device) * tau
    t_final.requires_grad = True
    final_loss = model(A_idx, t_final) - f_final

    # 总损失
    final_loss_val = (
        pde_loss_val +
        boundary_loss_A0.pow(2).mean() +
        boundary_loss_A1.pow(2).mean() +
        2e1 * final_loss.pow(2).mean()
    )

    return final_loss_val


# 设置参数
torch.manual_seed(101)
device = "cuda" if torch.cuda.is_available() else "cpu"

pi = 100.0  # 参数 pi
beta = 0.5  # 参数 beta
alpha = 0.8  # 参数 alpha
r = 0.05    # 无风险利率
T = 4.0     # 资产价格相关参数
tau = 3.0   # 期权到期时间
# K = 80.0    # 行权价

# 生成时间和资产价格网格
A_domain = [0.0, 100]
t_domain = [0.0, tau]
n_points_A = 250
n_points_t = 250
A_idx = torch.linspace(A_domain[0], A_domain[1], steps=n_points_A, requires_grad=True)
t_idx = torch.linspace(t_domain[0], t_domain[1], steps=n_points_t, requires_grad=True)
grids = torch.meshgrid(A_idx, t_idx, indexing="ij")
A_idx, t_idx = A_idx.reshape(-1, 1).to(device), t_idx.reshape(-1, 1).to(device)
A, t = grids[0].flatten().reshape(-1, 1).to(device), grids[1].flatten().reshape(-1, 1).to(device)

# 定义行权价范围
K_values = range(10, 91, 10)  # 从10到90，步长为10

# 创建存储结果的文件夹
import os
os.makedirs("pi_K", exist_ok=True)

# 遍历不同的行权价
for K in K_values:
    print(f"Processing K={K}...")
    
    # 初始化神经网络
    model = simple_NN2().to(device)

    # 优化器
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # 训练循环
    train_loss = []
    for ep in range(15001):
        loss = compute_loss(
            model, A, t, A_idx, t_idx,
            pi=pi, beta=beta, alpha=alpha, r=r, T=T, tau=tau, K=K, device=device
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss.append(loss.item())
        if ep % 300 == 0:
            print(f"Epoch: {ep}, Loss: {loss.item():.6f}")

    # 提取预测结果
    y = model(A, t)
    sol_pinn_matrix = torch.cat([A, t, y], dim=1)
    
    # 保存结果到文件
    filename = f"pi_K/sol_pinn_matrix_K_{K}.npz"
    np.savez(filename, sol_pinn_matrix=sol_pinn_matrix.cpu().detach().numpy())
    print(f"Results saved to {filename}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
# 设定要绘制的K值列表
K_values = [10, 20, 30, 40, 50, 60, 70, 80, 90]
tau = 3.0   # 期权到期时间

# 创建一个 2x4 的图窗
fig, axes = plt.subplots(3, 3, figsize=(16, 8))  # 2 行 × 4 列的网格

# 将 axes 展平，方便逐个操作
axes = axes.flatten()

# 遍历不同的K值
for i, K in enumerate(K_values):
    
    # 加载对应K值的结果
    filename = f"pi_K/sol_pinn_matrix_K_{K}.npz"  # 请根据实际文件名调整
    data = np.load(filename)
    sol_pinn_matrix = data['sol_pinn_matrix']
    
    # 提取预测结果
    A = sol_pinn_matrix[:, 0] # Asset price
    t = sol_pinn_matrix[:, 1] # Time
    C = sol_pinn_matrix[:, 2] # Option price

    # 确定绘图的时间点 t（以 0.5 为间隔）
    time_points = np.arange(0, tau + 0.1, 0.5)

    ax = axes[i]  # 获取当前子图
    for t_fixed in time_points:
        # 提取当前时间点的价格和资产价格
        indices = np.isclose(t, t_fixed, atol=1e-2)  # 找到接近 t_fixed 的点
        A_at_t = A[indices]
        C_at_t = C[indices]

        # 绘制曲线
        ax.plot(A_at_t, C_at_t, label=f"t = {t_fixed:.1f}")

    ax.set_xlabel("Asset Price (A)")
    ax.set_ylabel("Option Price (C)")
    ax.set_title(f" (K = {K})")
    ax.legend()
    ax.grid(True)


# 调整子图之间的间距
plt.tight_layout()
plt.show()


In [ ]:

# 创建一个 3x3 的图窗
fig, axes = plt.subplots(3, 3, figsize=(16, 8)) 

# 将 axes 展平，方便逐个操作
axes = axes.flatten()

# 遍历不同的K值
for i, K in enumerate(K_values):
    # 加载对应K值的结果
    filename = f"pi_K/sol_pinn_matrix_K_{K}.npz"  # 请根据实际文件名调整
    data = np.load(filename)
    sol_pinn_matrix = data['sol_pinn_matrix']
    
    # 提取预测结果
    A = sol_pinn_matrix[:, 0] # Asset price
    t = sol_pinn_matrix[:, 1] # Time
    C = sol_pinn_matrix[:, 2] # Option price

    # 确定绘图的时间点 t（以 0.5 为间隔）
    time_points = np.arange(0, tau + 0.1, 0.5)

    # 第二列：二维势能面
    ax = axes[i]
    
    # 创建网格
    A_range = np.linspace(A.min(), A.max(), 200)
    t_range = np.linspace(t.min(), t.max(), 200)
    A_grid, t_grid = np.meshgrid(A_range, t_range)

    # 插值计算网格上的期权价格
    C_grid = griddata((A, t), C, (A_grid, t_grid), method='cubic')

    # 绘制二维势能面
    contour = ax.contourf(A_grid, C_grid, t_grid, levels=50, cmap="viridis")

    # 添加颜色条
    cbar = plt.colorbar(contour, ax=ax)
    cbar.set_label("Time (t)")

    ax.set_xlabel("Asset Price (A)")
    ax.set_ylabel("Option Price (C)")
    ax.set_title(f"2D Potential Surface of Option Price (K = {K})")
    ax.grid(True, linestyle="--", alpha=0.5)

# 调整子图之间的间距
plt.tight_layout()
plt.show()


In [ ]:


# 设定要绘制的K值列表
K_values = [10, 20, 30, 40, 50, 60, 70, 80, 90]
tau = 3.0   # 期权到期时间

# 设定固定的 t 时刻（取 9 个等间距点）
t_fixed_points = np.arange(0, tau + 0.1, 0.5)

# 创建一个 3x3 的图窗
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()  # 展平子图数组，便于按顺序访问

# 遍历每个固定的 t 时刻
for i, t_fixed in enumerate(t_fixed_points):
    ax = axes[i]
    
    # 对每个 t 固定时刻，遍历所有不同的 K
    for K in K_values:
        # 加载对应 K 值的结果
        filename = f"pi_K/sol_pinn_matrix_K_{K}.npz"
        data = np.load(filename)
        sol_pinn_matrix = data['sol_pinn_matrix']
        
        # 提取预测结果：资产价格 A, 时间 t, 期权价格 C
        A = sol_pinn_matrix[:, 0]
        t = sol_pinn_matrix[:, 1]
        C = sol_pinn_matrix[:, 2]
        
        # 筛选出 t 值接近当前固定时刻 t_fixed 的数据（容差可根据实际情况调整）
        indices = np.isclose(t, t_fixed, atol=1e-2)
        A_at_t = A[indices]
        C_at_t = C[indices]
        
        # 若存在数据则绘制曲线
        if A_at_t.size > 0 and C_at_t.size > 0:
            ax.plot(A_at_t, C_at_t, label=f"K = {K}")
    
    # 设置子图标签、标题和图例
    ax.set_xlabel("Asset Price (A)")
    ax.set_ylabel("Option Price (C)")
    ax.set_title(f"t = {t_fixed:.2f}")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()
